# Everything as a Table — SQL as the orchestration language for SPUR

**Status:** draft (design spec, code-grounded update 2026-06-11) · **Date:** 2026-06-08 · **Owner area:** `crates/spur-context` + `crates/spur-notebook/rest-table-gateway` + `crates/spur-notebook/rest-table-gateway-ext`

**One-line thesis:** every SPUR capability — read, write, search, analyze, dispatch, observe — becomes a **DuckDB table function** or **action surface**. SQL stops being *a way to query data inside the notebook* and becomes **the unified control plane for SPUR itself**: the brain, the human in a notebook, and the TUI all speak the same dialect against the same live catalog.

> Brainstormed decisions locked in this spec: read + write surfaces both first-class · SQL is the control plane, not just the data plane · `mcp_tools()` + `mcp_call()` (no per-tool auto-typed vtables in v1) · `knowledge_context_pack()` as a first-class TVF · gateway is the plug-in point, not the destination · shared DuckDB instance (one catalog, not two) · catalog drift handled per-`bind()` re-fetch with TTL fallback.

**Grounding note (2026-06-11):** the thesis still holds, but current code is further along in the extension crate and less far along in the shared-catalog layer than the original draft implied. The base `spur-rest-table-gateway` crate already owns the `Adapter` trait, `TableKind::{Table, TableFunction, Action}`, `IoBridge`, and the zero-arg table-function path. The load-bearing parameterized table-function and action registration path already exists in `crates/spur-notebook/rest-table-gateway-ext`, but it is still specialized and should be hardened/generalized before new sources depend on it. The shared DuckDB instance does **not** exist yet: `spur-context`, `spur-analyst`, notebook datasource probing, and Python setup cells currently open separate connections. P1 is therefore a real ownership/lifecycle migration, not a small pointer-sharing patch.

This spec is normative. Diagrams are normative; prose is supporting. The new work is *coverage* (more sources), *generalization* (turn the extension's parameterized/action path into a source-agnostic gateway surface), and *connection ownership* (one catalog visible to notebook/context/analyst), not invention from scratch.

## 1. Vision — one surface, three audiences

The same SQL statement is consumed by three different call sites, all of which already exist in SPUR today. The spec is the **third user** of an idea whose first two are already shipping: the gateway is the substrate, the notebook is the human surface, the brain is the agent surface.

```mermaid
flowchart LR
    subgraph SOURCES["Sources (Adapter trait)"]
        REST["rest-table-gateway<br/>(production — Polymarket, etc.)"]:::keep
        MCP["mcp-client (NEW)<br/>tools · resources · prompts"]:::new
        CODE["spur-context<br/>code · plans · sessions"]:::new
        BEADS["beads-rust<br/>issues · epics · deps"]:::new
        GIT["spur-worktree<br/>commits · diffs · status"]:::new
        COST["spur-cost<br/>sessions · ledger"]:::new
    end

    subgraph DUCK["Single shared DuckDB instance"]
        CAT["Catalog:<br/>table funcs · vtables · actions"]
        TVF["zero-arg TVFs<br/>&lt;source&gt;_&lt;thing&gt;()"]
        PARA["parameterized TVFs<br/>&lt;source&gt;_&lt;thing&gt;(arg...)"]
        ACT["actions<br/>mcp_call · beads_create_issue"]
        CAT --> TVF --> PARA
        CAT --> ACT
    end

    SOURCES --> CAT

    subgraph CONSUMERS["Consumers"]
        NB["Jute notebook<br/>(SQL cells)"]
        BRAIN["Brain agent<br/>(ACP + plan engine)"]
        TUI["spur-tui<br/>(session picker, log view)"]
    end

    DUCK --> NB
    DUCK --> BRAIN
    DUCK --> TUI

    classDef new fill:#dcfce7,stroke:#16a34a,color:#14532d
    classDef keep fill:#f1f5f9,stroke:#94a3b8,color:#334155
```

**The radical move:** the brain, today, hand-codes its dispatch logic in Rust. Tomorrow it writes a query. `WITH tool AS (SELECT name FROM mcp_tools() WHERE description ILIKE '%github%') INSERT INTO mcp_action … RETURNING …` — same outcome, but it's a *declarative* plan the planner can optimize, log, replay, and the user can inspect from a notebook cell next door. The brain stops being a closed loop; it becomes a query that other queries can compose with.

**The less radical but more durable move:** the notebook stops being a downstream consumer of SPUR and becomes a *control plane for SPUR*. A user opens a notebook, types `SELECT * FROM knowledge_context_pack('how do I add a tool', 'plan')` to teach themselves the system, then `INSERT INTO delegate_to_worker(...)` to actually do the work. The same notebook is documentation, telemetry, and ops console.

## 2. The matrix — what becomes a table, what becomes an action

The `TableKind` enum in `spur-rest-table-gateway/src/adapter/mod.rs:119–131` already distinguishes `Table`, `TableFunction { arg_names }`, and `Action`. That taxonomy maps onto every SPUR capability with no new types. The matrix below is the spec's *coverage* — what gets built, in what order.

| Surface | Read (`SELECT * FROM ...`) | Write (`INSERT INTO ...`) | Source adapter | Phase |
|---|---|---|---|---|
| REST / GraphQL APIs | `polymarket_markets()` | `polymarket_place_order(...)` | `rest-table-gateway` (exists) | **shipped** |
| MCP tools (catalog) | `mcp_tools()` (built-in TVF) | — | `spur-mcp-client` (NEW) | P1 |
| MCP tools (call) | `mcp_call(server, tool, args_json)` | `mcp_call_action` | `spur-mcp-client` | P1 |
| MCP resources | `mcp_list_resources(server)` + `mcp://server/uri` URI scheme | — | `spur-mcp-client` | P1 |
| Code graph (search) | `code_symbol_search('delegate')` | — | `spur-context` (NEW TVF) | P1 |
| Code graph (read) | `code_read_symbol('path.rs::fn')` | — | `spur-context` | P1 |
| Knowledge context pack | `knowledge_context_pack('query', 'debug')` | — | `spur-context` → `spur-analyst` BM25 | P1 |
| Beads / issues | `beads_list_issues(filter)` | `beads_create_issue(...)` returning id | `spur-pm` (NEW source) | P2 |
| Beads / epics + DAG | `beads_epic_dag(epic_id)` | `beads_execute_epic(epic_id)` | `spur-pm` | P2 |
| Sessions | `sessions()` joining `spur-cost` ledger | `delegate_to_worker(...)` | `spur-cost` + `spur-mcp` | P2 |
| Plans | `plan_status(plan_id)` | `plan_review(task_id, decision)` | `spur-mcp` | P2 |
| Git / worktree | `git_status(path)` | `git_commit(...)` | `spur-worktree` | P3 |
| Files (read-only) | `fs_list(path)`, `fs_read(path)` | — | `spur-notebook` (datasource path) | P3 |
| **Per-tool typed vtable** (e.g. `github_list_issues(state)`) | per-tool, with typed `outputSchema` | same | `spur-mcp-client` | **deferred** |

**Why per-tool typed vtables are deferred, not v1:** they require the MCP server to ship a non-trivial `outputSchema` (most don't, per audit of `modelcontextprotocol/servers` reference impls), they multiply catalog entries by N tools × M servers, and they break the moment a server revises a schema. The escape-hatch `mcp_call(server, tool, args_json)` covers every tool regardless of schema quality. The typed projection is a power-user affordance for a future phase, gated on catalog drift measurements in P2.

## 3. The `Adapter` trait already generalizes — leverage it, but account for the extension split

`spur-rest-table-gateway/src/adapter/mod.rs:141–150` defines `trait Adapter { fn name() -> &str; fn catalog() -> Vec<TableDef>; async fn scan(ScanRequest) -> Result<Vec<RecordBatch>>; async fn act(ActionRequest) -> ... }`. Every new source is just another `impl Adapter`. The hard parts are partly solved:

- **Sync→async bridge** — `IoBridge` (`vtab/bridge.rs`) owns a single-thread tokio runtime behind a `mpsc::Sender<Job>`; it accepts `Arc<dyn Adapter>` so any registered adapter's `scan()` runs from DuckDB's sync thread without blocking the engine. It already has `Job::Act` and `call_act()`, so action dispatch is not hypothetical.
- **Arrow→DuckDB vector writer** — `vtab/table_fn.rs` maps `Utf8 | Int64 | Float64 | Boolean` into `DataChunkHandle::flat_vector`; extensions land in a single match arm.
- **Base registration** — `vtab/register.rs:12–36` still registers only `TableKind::Table` as `<source>_<table>()`. `ApiTableVTab::parameters()` still returns `None`; the base crate does not yet expose source-agnostic parameterized table functions.
- **Extension registration** — `rest-table-gateway-ext/src/lib.rs::register_adapter` already handles `Table`, `TableFunction { arg_names }`, and `Action`. This is the concrete implementation to harden/upstream, not a future invention.
- **Toml manifest driver** — `adapter/manifest.rs` deserializes `[[table]]` and `[[action]]` blocks, so OpenAPI/Nango imports flow through the same `Adapter` surface.

The key P0 work is to reconcile the base crate and extension crate: promote the extension's `ApiFunctionVTab`/`ApiActionVTab` path into a source-agnostic gateway surface, remove Polymarket-specific argument handling (`token_id`/`depth` hard-coding), and make `arg_names`/`ArgSpec` drive DuckDB named parameters generically. Everything after that can build adapters instead of fighting the vtab substrate.

## 4. Component architecture — what lands where

New crates and surfaces, with the boundary rule preserved (no upstream `jute` coupling; only additive `metadata.spur.*`).

```mermaid
flowchart TB
    subgraph NB[crates/spur-notebook]
        GW["rest-table-gateway<br/>(existing) — Adapter trait, IoBridge,<br/>zero-arg ApiTableVTab"]:::keep
        EXT["rest-table-gateway-ext<br/>(existing) — TableFunction + Action<br/>registration path"]:::keep
        GW2["P0 hardening<br/>generic named args + source-agnostic actions<br/>upstream/reconcile base + ext"]:::edit
    end

    subgraph NEWC[NEW crates]
        MCPC["spur-mcp-client<br/>MCP transport (stdio/http/sse)<br/>+ tools/resources/prompts Adapter"]:::new
        PMSC["spur-pm-source<br/>beads-rust Adapter (issues, epics, DAG)"]:::new
    end

    subgraph EXIST[crates/spur-context — edit]
        CODEG["code_* TVFs (search/read/history)"]:::edit
        KCP["knowledge_context_pack TVF<br/>→ current MCP handler + spur-analyst BM25"]:::edit
    end

    subgraph DUCK[Shared DuckDB instance]
        CONN["NEW ownership layer:<br/>one catalog visible across<br/>notebook + spur-context + spur-analyst"]:::new
        REGFN["registry.register(adapter)<br/>→ conn.register_table_function_with_extra_info"]:::keep
    end

    GW --> GW2
    EXT --> GW2
    GW2 --> CONN
    MCPC --> CONN
    PMSC --> CONN
    CODEG --> CONN
    KCP --> CONN
    REGFN --> CONN

    subgraph CON[Consumers]
        CELLS["Jute notebook SQL cells / setup cells"]
        BRN["Brain agent<br/>(compose queries in plan)"]
        TUI2["spur-tui<br/>(read-only views into catalog)"]
    end

    CONN --> CELLS
    CONN --> BRN
    CONN --> TUI2

    classDef new fill:#dcfce7,stroke:#16a34a,color:#14532d
    classDef edit fill:#fef9c3,stroke:#ca8a04,color:#713f12
    classDef keep fill:#f1f5f9,stroke:#94a3b8,color:#334155
```

**Critical decision: one shared DuckDB.** The matrix in §2 is impossible if `spur-context`'s DuckDB, `spur-analyst`'s DuckDB, and the notebook's DuckDB are three separate connections — `SELECT * FROM mcp_tools() JOIN knowledge_context_pack(...)` is then three separate engines with no shared planner.

**Current code reality:** they are separate today. `spur-context::AnalyticsEngine` owns `conn: Connection` and opens persistent/in-memory DuckDB itself. `spur-analyst` opens read-only connections from a `db_path` per query path. Notebook datasource introspection opens probe connections, and the notebook setup preamble creates a Python in-memory default connection. The current notebook can bootstrap DuckDB views and load the REST extension, but that is not yet the unified live catalog this spec proposes.

The fix is an explicit DuckDB ownership layer, not incidental `Arc` threading. v1 still targets **one connection per process**, likely behind `Arc<Mutex<Connection>>` or an actor if contention shows up, but P1 must define: who owns the connection, how notebook/kernel setup reaches it, how `spur-analyst` either attaches or registers its BM25 views into it, and how lifecycle/re-registration works. ATTACH-chained catalogs remain acceptable if they preserve single-query joins; two unrelated engines are not.

## 5. Write surfaces — partially built, still needs policy and generalization

DuckDB does not have first-class `INSERT INTO <table-function>`, but the gateway already has the more useful abstraction: `TableKind::Action` plus `Adapter::act(ActionRequest) -> Vec<RecordBatch>`. Current code also already has `IoBridge::call_act()` and an `ApiActionVTab` in `rest-table-gateway-ext` that composes an `ActionRequest`, invokes the adapter, and returns rows.

Three implementation strategies, in order of preference:

1. **Use the existing `TableKind::Action` path as the v1 write surface.** Surface actions as table/scalar-call shapes such as `SELECT * FROM mcp_call_action(server := 'github', tool := 'list_issues', args_json := '{...}')` or source-specific action functions. This reuses `act(adapter, ActionRequest) -> Vec<RecordBatch>` and works with normal query composition.
2. **Generalize action named parameters.** The extension already has dynamic named parameters for actions via `ArgSpec`; harden this path and enforce `allow_writes`, idempotency, dry-run, and human-confirm policy before MCP/beads actions are exposed.
3. **`INSERT INTO <vtab>` with `Inserter`.** Keep this as a v2 affordance for SQL users who strongly want `RETURNING` syntax. It is not required for v1 because table-function actions can already return a row batch.
4. **Per-source stateful in-process queue.** For streaming or fire-and-forget actions such as `delegate_to_worker`, return an id immediately and persist later progress elsewhere. Defer to v2.

**v1 uses strategy 1 + 2.** The work is not inventing writes; it is making the existing extension action path source-agnostic, policy-gated, and testable.

## 6. Worked example — the brain, but as a query

A representative brain loop today (pseudocode, brain's Rust): `read config → match name → look up handler → call handler → format result → return`. As a query, with this spec's surfaces in place:

```sql
-- 1. Discover: what MCP tools match this user's intent?
WITH tool AS (
  SELECT server, name, description
  FROM mcp_tools()
  WHERE description ILIKE '%github%' OR description ILIKE '%jira%'
  ORDER BY rank  -- future: BM25 over description
  LIMIT 5
)
-- 2. Read context: what does the codebase say about delegation failure?
, kcp AS (
  SELECT symbol, file_path, score
  FROM knowledge_context_pack('delegation failure handling', 'debug')
  LIMIT 3
)
-- 3. Issue: pull open issues to triage
SELECT t.server, t.name AS tool, kcp.symbol AS context_hit
FROM tool t, kcp
WHERE EXISTS (
  SELECT 1 FROM mcp_call(t.server, t.name, '{"state":"open","limit":5}') result
  WHERE json_extract_string(result.content, '$.total_count') > 0
);
```

This is **one planner pass** over live MCP servers, a BM25 index, and the catalog. Today it's three Rust functions, three config files, and a hand-written JSON-RPC loop. The shape of the brain *changes* — from imperative to declarative — and every consumer (notebook, brain, TUI) gets the same answer.

## 7. Data model — the catalog, the schemas, the policy

Three orthogonal data shapes live in the spec: the **catalog** (which table functions exist), the **schemas** (their column lists), and the **policy** (who can invoke which action).

```mermaid
classDiagram
    direction LR
    class Adapter {
        <<trait, in spur-rest-table-gateway>>
        +name() String
        +catalog() Vec~TableDef~
        +async scan(ScanRequest) Vec~RecordBatch~
        +async act(ActionRequest) Vec~RecordBatch~
    }
    class TableDef {
        +name String
        +schema SchemaRef
        +kind TableKind
    }
    class TableKind {
        <<enum>>
        Table
        TableFunction arg_names
        Action method, path, arg_specs
    }
    class McpTableDef {
        +server String
        +tool String
        +input_schema JSON
        +output_schema Option~JSON~
    }
    class McpAction {
        +server String
        +tool String
        +args_json JSON
        +idempotency_key Option~String~
    }
    class CatalogPolicy {
        +allow_writes bool
        +allowed_servers HashSet~String~
        +rate_limit PerMinute
    }
    Adapter ..> TableDef : produces
    TableDef *-- TableKind
    McpTableDef --|> TableDef
    McpAction ..> CatalogPolicy : gated by
    CatalogPolicy ..> McpAction : enforced at act()
```

**Catalog policy is mandatory, not optional.** The REST gateway already has `allow_writes` in `SourceCfg` (`adapter/manifest.rs:33`); MCP sources inherit the same gate. `INSERT INTO mcp_call_action(server='github', tool='delete_repo', ...)` is a one-line catastrophic action; the policy must (a) require `allow_writes = true` on the source, (b) be human-confirmable at the call site, mirroring the MCP spec's own human-in-the-loop guidance, and (c) carry a per-source rate limit. v1 ships policy as a `RUST_POLICY` constant; v2 reads from notebook metadata so users can override per-notebook.

## 8. Catalog drift — the actual hard problem

`mcp_tools()` returns whatever the connected server advertises *at the moment of the query*. If the server revises a tool's `inputSchema` between connection and bind, the vtable's column list lies. The gateway's `bind()` is called once per query plan; the strategies, in order of cost:

1. **Per-`bind()` re-fetch.** Expensive (one round-trip per query) but always-correct. Use for `mcp_tools()` itself (small, fast) and for `mcp_call` (just a JSON column).
2. **TTL-cached catalog with `list_changed` subscription.** Re-fetch every N seconds OR on the MCP `notifications/tools/list_changed` notification. The transport layer subscribes; on notification, increment the catalog's `schema_version` and unregister/re-register the affected TVFs. v1 default: TTL 60s, instant refresh on notification.
3. **Schema-pinned per-query.** The query declares its expected schema (`SELECT * FROM mcp_call('server', 'tool', args) WITH (expected_schema = '...')`). If the live schema mismatches, error. v2.

**v1 ships #1 + #2** for `mcp_tools()`. `mcp_call()` always re-fetches the schema (it's one JSON column — no vtable). Per-tool typed vtables (the deferred row in the matrix) are the only place where drift bites, and they're not in v1.

## 9. Non-negotiable invariants (inherited)

1. **The boundary rule.** New code lands in `spur-notebook/rest-table-gateway`, `spur-mcp-client`, `spur-pm-source`, and `spur-context` — never in `jute-notebook` itself. `jute` remains a generic, upstream-trackable library. The notebook surface gains exactly the additive `metadata.spur.sql_catalog` (a pointer to the shared connection's TVF list) needed for autocomplete; no new behaviors.
2. **One authoritative catalog per process.** A single `Arc<Mutex<Connection>>` shared by the notebook daemon, `spur-context`, and `spur-analyst`. ATTACH-chained is acceptable if crate boundaries demand it; two separate engines is not.
3. **Progressive enhancement.** All new TVFs are additive registrations on the existing `Connection` — no new file format, no new notebook kind. A vanilla `nbformat` notebook that doesn't know about `mcp_tools()` simply fails at query time with a clean error, not at parse time.
4. **No double-subscribe on the shared connection.** The catalog re-registration path is `unregister(name); register(name, schema)` — idempotent, atomic per TVF. The MCP `list_changed` handler re-registers; a parallel query sees the new schema on its next bind.
5. **Human-in-the-loop on writes.** Every action surface honors the source's `allow_writes` flag and emits a confirmation prompt in the TUI/notebook. The brain's plan engine may bypass for documented autonomous paths (e.g. `delegate_to_worker` is a delegating *read* in v1; the actual worker is what writes, and the worker's `spur-mcp` MCP tools already carry their own confirmation policy).
6. **Fail loud, not silent.** A TVF whose bind fails emits `set_error` on `BindInfo` (the gateway's `ApiTableVTab` already does this on `arrow_to_duckdb_type` mismatches). The notebook UI shows the error in the cell output; the brain receives a structured error and may emit a `signal:scope-drift`.

## 10. v1 scope (YAGNI) and the things explicitly deferred

| In v1 | Deferred (designed-for, not built) |
|---|---|
| `mcp_tools()` TVF (per-server, live) | Per-tool typed vtables (`github_list_issues(state)`) |
| `mcp_call(server, tool, args_json)` scalar | Streaming / long-poll MCP responses |
| `mcp_list_resources(server)`, `mcp_list_prompts(server)` TVFs | `mcp://<server>/<uri>` URI scheme for resources |
| `code_symbol_search`, `code_read_symbol`, `code_symbol_history` TVFs | `code_subgraph` (radius>1) — too expensive per-bind |
| `knowledge_context_pack(query, intent)` TVF | BM25 rank ordering inside `mcp_tools()` |
| `beads_list_issues`, `beads_create_issue` | `beads_execute_epic` (delegation to plan engine) |
| `git_status`, `git_diff` (read) | `git_commit`, `git_push` (write — needs human confirm) |
| Shared DuckDB connection (single instance, `Arc<Mutex<…>>`) | ATTACH-chained catalogs across crates |
| Catalog policy: `allow_writes` per source | Per-notebook policy overrides from metadata |
| TTL-cached catalog + `list_changed` push | Schema-pinned per-query (`WITH (expected_schema)`) |
| Brain reads via existing `spur-mcp` tools (no brain change in v1) | Brain writes SQL plans against this catalog |

**Acceptance test for v1:** a user can open a notebook, run `SELECT * FROM mcp_tools()` against a connected GitHub MCP server, run `SELECT * FROM knowledge_context_pack('X', 'debug')`, run `SELECT * FROM beads_list_issues()` — and JOIN the three. The brain does not need to learn this surface to ship v1; the value proves itself in the notebook first.

## 11. Build sequence — phased, dependency-ordered, each phase shippable

```mermaid
flowchart LR
    P0["P0 — gateway substrate hardening<br/>generalize rest-table-gateway-ext TableFunction/Action<br/>remove token_id/depth special cases"]:::edit
    P1["P1 — shared DuckDB ownership<br/>single catalog connection or ATTACH-equivalent<br/>across notebook + context + analyst"]:::new
    P2["P2 — code_* TVFs<br/>code_symbol_search · read_symbol · symbol_history<br/>+ knowledge_context_pack"]:::new
    P3["P3 — spur-mcp-client crate<br/>MCP transport + Adapter impl<br/>+ mcp_tools() + mcp_call()"]:::new
    P4["P4 — beads source<br/>spur-pm-source Adapter<br/>+ beads_list_issues + beads_create_issue"]:::new
    P5["P5 — catalog policy<br/>allow_writes · allowlist · rate limit<br/>+ human-confirm on writes"]:::new
    P6["P6 — list_changed re-registration<br/>MCP push subscription + unreg/rereg on change"]:::new
    P7["P7 — notebook UI<br/>SQL cell/setup integration, autocomplete over catalog,<br/>output panel renders query results"]:::edit
    P0 --> P1 --> P2 --> P3 --> P4 --> P5 --> P6 --> P7
    classDef new fill:#dcfce7,stroke:#16a34a,color:#14532d
    classDef edit fill:#fef9c3,stroke:#ca8a04,color:#713f12
```

**P0 touches shipped code, but starts from existing extension behavior.** It is not a fresh ~50-LOC implementation in `table_fn.rs`/`register.rs`; it is a hardening/reconciliation pass over `rest-table-gateway` and `rest-table-gateway-ext`. The acceptance target is a manifest-defined parameterized table function whose argument names and types are not hard-coded to Polymarket, plus an action that routes through `Adapter::act()` with policy hooks available.

**P1 is the first architectural migration.** Today, `spur-context`, `spur-analyst`, notebook datasource probing, and Python notebook setup open separate DuckDB connections. P1 must introduce the owner/access pattern for the shared catalog and prove cross-source joins on one planner-visible connection.

**Per-phase validation gate:** each phase ships a notebook cell or Rust integration test that exercises the new surface end-to-end. P0: a manifest with a generic parameterized table function and action. P1: notebook/context/analyst query through one catalog or ATTACH-equivalent. P2: a `code_*` + `knowledge_context_pack` join. P3: a live MCP `mcp_tools()` query. P4: a beads create action returning an id. P5: a denied write with a clean error. P6: a `list_changed` refresh observable on the next bind. P7: a SQL cell/setup path in the notebook UI with autocomplete.

## 12. Risks — the things that could sink this

| Risk | Likelihood | Mitigation |
|---|---|---|
| **Three DuckDB connections, not one** — `spur-context`, `spur-analyst`, notebook datasource probing/setup each open their own today. The spec's premise (joins across sources) fails unless P1 changes ownership. | High | P1's acceptance test asserts that notebook-visible SQL can join at least one context/analyst surface and one gateway-registered surface in a single query plan. Use `Arc::ptr_eq` only if the design is literally shared-connection; use an ATTACH visibility assertion if that is the chosen route. |
| **Extension/base split hides substrate bugs** — base `register_tables` only handles `Table`; extension handles `TableFunction`/`Action` but with special-case argument wiring. | High in P0 | P0 hardens the extension path first, removes `token_id`/`depth` special cases, and decides whether to upstream into the base gateway or keep the extension as the registration owner. |
| **MCP servers that don't ship `outputSchema`** — most don't, so the catalog row for a tool has a `null` `output_schema` and a `JSON` typed call. | Certain | This is why v1 picks the catalog + JSON call pattern. The escape hatch works; typed projection is deferred. |
| **Catalog drift mid-query** — server revises schema between `bind` and `func`. | Medium | `bind()` re-fetches the catalog every call for cheap catalog surfaces; TTL for the long-tail. v2 pins per-query. |
| **Brain bypasses SQL and goes back to Rust** — if writing SQL is harder than Rust, the brain never adopts. | High if v1 is awkward | P7's notebook UI is the canary. If humans don't write SQL there, the brain won't either. Re-evaluate at the end of P7. |
| **Action surface becomes a security hole** — `mcp_call_action` deletes repos. | Certain without gates | P5's policy gate is non-negotiable; `allow_writes = false` is the default; human confirmation mandatory in notebook and TUI. |
| **Streaming tools don't fit** — MCP servers that emit `notifications/progress` mid-call. | Medium | v1 collects all chunks into a single batch; v2 designs the streaming path against DuckDB's evolving support. Documented as out-of-scope. |
| **Per-source rate limits become a hot path** — every action checks. | Low in v1 | `tokio::sync::Semaphore` per source; cap at 10 r/s default. Cheap to add later. |
| **Notebook UI doesn't render query results** — `OutputView` is built around notebook MIME bundles; RecordBatch rendering exists in port/bootstrap paths but not necessarily as a first-class SQL result view. | Medium | Either serialize query results to `text/html` table output first, or ship an Arrow-aware output view. P7 picks one. |
| **`IoBridge` is single-threaded** — concurrent TVF scans serialize. | Low in v1 (one user per notebook session) | Replace `mpsc::channel` with a small worker pool in v2 if profiling shows contention. |
| **The brain doesn't actually want to write SQL** — maybe the Rust path is genuinely simpler. | High | This is the test of the spec. If after P7 a brain agent consistently prefers Rust handlers over the SQL surface, the unification is not paying for itself. The honest answer may be "no." |

## 13. Success criteria — when do we know this worked?

Three measurable signals, in order of importance:

1. **A human in a notebook can answer "what tools does SPUR know about right now?" with one query.** `SELECT * FROM mcp_tools() JOIN beads_list_issues() ON …` returns a live answer in <1s. If this works, the unification is real.
2. **A brain plan that previously was 50 lines of Rust becomes 10 lines of SQL.** Audit one concrete plan pre- and post-adoption. If the SQL version is *not* shorter, the abstraction isn't paying for itself.
3. **A new MCP integration takes <1 day instead of <1 week.** Measure end-to-end from "new MCP server available" to "queryable in notebook." The 5th server is the canary — the 1st will take the same time, the 5th should be hours.

If signal 1 is met but signals 2 and 3 are not, ship it as a notebook feature anyway and revisit the brain adoption in v2. Don't let perfect block shippable.

## 14. Open questions / future work

- **Brain-as-query-rewriter.** If the spec succeeds, the brain's planning loop becomes "given a goal, write the SQL." That's a real new capability — and a real new failure mode. v2 designs the planner-aware prompt and the validation harness.
- **Streaming TVF support.** DuckDB's evolving. v1 is one-shot; revisit when the API stabilizes.
- **Cross-source joins with predicate pushdown.** Today, `mcp_call()` returns the whole result, then DuckDB filters. With typed per-tool vtables (deferred), pushdown becomes possible. Measure first; don't pre-build.
- **Catalog policy in notebook metadata.** v1 is code-only; v2 makes `allow_writes` overridable per-notebook so a "read-only sandbox" notebook can be checked in.
- **SQL cell as the notebook's first-class cell type.** If the catalog grows, a `%%sql` cell magic (Python kernel) and a dedicated SQL cell (Deno kernel) are obvious next steps. The `notebook-polyglot-cell-ui-design` spec from 2026-06-03 covers the kernel side; this spec covers the catalog side. They compose.
- **ATTACH-chained vs. shared connection.** v1 ships a shared connection. If the spur-analyst BM25 index genuinely needs its own engine (e.g. different DuckDB version), ATTACH is the escape hatch. Measure first.
- **What is the right v1 demo?** A notebook that does `SELECT * FROM mcp_tools() WHERE description ILIKE '%github%'` then `SELECT * FROM mcp_call('github', 'list_issues', '{"state":"open"}')` then `SELECT * FROM beads_create_issue(title, body) RETURNING id` — the whole loop in three cells. That's the 30-second sales pitch for the architecture.

---

**Next step:** transition to the `writing-plans` skill to produce the phased implementation plan (P0–P7). P0 should start with the current `rest-table-gateway-ext` implementation: generalize parameterized table-function argument binding, harden action registration/policy hooks, and decide whether that code moves into the base gateway crate or remains the extension's responsibility. Do that before opening new MCP/beads/code source crates.